# OPRA Policy Evaluation

Evaluates DRL-AR-VDDPS simulated logs against the BPCI_2012 DDPS reference log.

In [ ]:
import sys
sys.path.insert(0, 'src')

import glob
from metrics.evaluation_metrics import PolicyEvaluator, aggregate_results

In [ ]:
ORIGINAL_LOG_PATH = 'data/simulated_logs/BPCI_2012/BPCI_2012_DDPS.csv'
ORIGINAL_LOG_NAMES = {'case': 'case', 'start': 'start', 'end': 'end'}

SIM_LOG_DIR = 'data/evaluation_results/DRL-AR/BPIC_2012/simulated_logs'
sim_log_paths = sorted(glob.glob(f'{SIM_LOG_DIR}/sim_run_*.csv'))

print(f'Found {len(sim_log_paths)} simulated logs:')
for p in sim_log_paths:
    print(f'  {p}')

In [ ]:
evaluator = PolicyEvaluator(
    original_log_path=ORIGINAL_LOG_PATH,
    original_log_names=ORIGINAL_LOG_NAMES,
    sla_percentiles=[95, 90, 75, 50],
)

In [ ]:
print('--- Per-run results ---')
per_run_perf = []
per_run_sim = []

for path in sim_log_paths:
    run_name = path.replace('\', '/').split('/')[-1].replace('.csv', '')
    perf, sim = evaluator.evaluate_single_log(path)
    per_run_perf.append(perf)
    per_run_sim.append(sim)

    print(f'
[{run_name}]')
    print(f'  Cases: {perf.num_cases}')
    print(f'  Avg CT: {perf.avg_cycle_time:.1f}s  |  Median CT: {perf.median_cycle_time:.1f}s')
    for label, cr in perf.compliance_rates.items():
        cir = perf.compliance_improvement_ratios[label]
        print(f'  {label}: CR={cr:.2%}  CIR={cir:+.2%}')
    if sim.ngd is not None:
        print(f'  Similarity -> NGD={sim.ngd:.4f}  AED={sim.aed:.4f}  CED={sim.ced:.4f}  '
              f'RED={sim.red:.4f}  CWD={sim.cwd:.4f}  CAR={sim.car:.4f}  CTD={sim.ctd:.4f}')

In [ ]:
agg = aggregate_results(per_run_perf, per_run_sim, policy_name='DRL-AR-VDDPS', log_name='BPIC_2012')
evaluator.print_results(agg)

In [ ]:
evaluator.save_results(agg, output_dir='data/evaluation_results')